### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 실습 (심화) — LangSmith로 자동 트레이싱·평가하기
**Production-Grade Agent Engineering · Colab 실습 (Upstage Solar × LangSmith)**

> 앞 실습에서 OpenTelemetry로 Span을 **직접** 계측했다면, 이번엔 관리형 플랫폼 **LangSmith**로 ① Solar 호출을 **자동 트레이싱**(코드 한 줄)해 trace 트리를 UI에서 보고 ② **데이터셋 + LLM-as-Judge evaluator**로 평가·실험 비교까지 한 번에 합니다.
>
> '직접 계측' vs '플랫폼이 다 해주는 방식'을 비교 체험하는 것이 목표입니다.
>
> 사전 준비: [smith.langchain.com](https://smith.langchain.com) 무료 가입 → API 키 발급 → Colab Secrets에 `LANGSMITH_API_KEY` 등록(기존 `UPSTAGE_API_KEY`도 필요).

## 0. 셋업
Colab 좌측 🔑 **Secrets**에 `UPSTAGE_API_KEY`와 `LANGSMITH_API_KEY` 두 개를 등록한 뒤 실행합니다.

In [10]:
!pip -q install langsmith openai

In [12]:
import os
from google.colab import userdata

# 이 환경변수 설정만으로 이후 모든 호출이 자동으로 LangSmith에 트레이싱됩니다.
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "agent-course-5강"

from openai import OpenAI
from langsmith import Client, traceable, evaluate
from langsmith.wrappers import wrap_openai

UPSTAGE_API_KEY = userdata.get("UPSTAGE_API_KEY")
MODEL = "solar-pro3"

# wrap_openai: Solar(OpenAI 호환) 클라이언트를 감싸면 호출이 LangSmith span으로 자동 기록
solar = wrap_openai(OpenAI(api_key=UPSTAGE_API_KEY, base_url="https://api.upstage.ai/v1"))
ls = Client()
print("LangSmith + Solar 준비 완료")

LangSmith + Solar 준비 완료


## 1. 자동 트레이싱 — OTel 수동 계측과 비교
앞 실습에서는 `with tracer.start_as_current_span(...)`로 Span을 일일이 감쌌습니다. LangSmith에서는 `@traceable` 데코레이터 하나면 끝입니다 — 내부 Solar 호출도 `wrap_openai` 덕에 **자식 span으로 자동 기록**되어, trace 트리가 UI에 그대로 펼쳐집니다.

In [15]:
NEWS = {
    "AI 반도체": ["A사 추론비용 40%↓", "B사 온디바이스 AI 칩 양산"],
    "전기차": ["전고체 배터리 시범 양산", "충전 표준 통합 논의"],
}

@traceable(name="search_news")          # 도구 호출도 span으로 기록
def search_news(keyword: str):
    for k, v in NEWS.items():
        if k in keyword or any(w in keyword for w in k.split()):
            return v
    return []

@traceable(name="news_agent")           # 루트 span
def run_agent(inputs: dict) -> dict:
    q = inputs["question"]
    kw = q.replace("동향", "").replace("요약", "").strip()
    hits = search_news(kw)
    resp = solar.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "검색된 헤드라인에만 근거해 한국어로 한 문장 요약. 근거 없으면 모른다고 답하라."},
        {"role": "user", "content": f"질문:{q}\n헤드라인:{hits}"}])
    return {"answer": resp.choices[0].message.content.strip()}

print(run_agent({"question": "전기차 배터리 동향 요약"}))
print("→ smith.langchain.com 프로젝트에서 trace 트리(에이전트→검색→LLM)와 토큰·지연을 확인하세요.")

{'answer': '전고체 배터리의 시범 양산과 충전 표준 통합 논의가 진행되고 있습니다.'}
→ smith.langchain.com 프로젝트에서 trace 트리(에이전트→검색→LLM)와 토큰·지연을 확인하세요.


## 2. 평가 데이터셋 만들기
관측만으로 끝이 아니라, 수집한 trace를 **평가**로 연결합니다. 입력/기대출력 쌍을 데이터셋으로 올립니다 (성공 + 함정 케이스 혼합).

In [16]:
DATASET = "5강-observability-eval"

examples = [
    {"inputs": {"question": "AI 반도체 동향 요약"},
     "outputs": {"expected": "추론 비용 절감·온디바이스 AI"}},
    {"inputs": {"question": "전기차 배터리 동향 요약"},
     "outputs": {"expected": "전고체 배터리 시범 양산"}},
    {"inputs": {"question": "C사 AGI 출시 소식 요약"},            # 존재하지 않음 = 함정
     "outputs": {"expected": "출처 없음 — 모른다고 답해야 함"}},
]

if ls.has_dataset(dataset_name=DATASET):
    print(f"데이터셋 이미 존재: {DATASET}")
else:
    ds = ls.create_dataset(dataset_name=DATASET, description="5강 관측성·평가 데이터셋")
    ls.create_examples(
        inputs=[e["inputs"] for e in examples],
        outputs=[e["outputs"] for e in examples],
        dataset_id=ds.id,
    )
    print(f"데이터셋 생성: {DATASET} ({len(examples)}건)")

데이터셋 이미 존재: 5강-observability-eval


## 3. LLM-as-Judge Evaluator 정의
5강에서 Span attribute로 직접 평가기를 짰듯, 여기서는 **evaluator 함수**로 정의하면 LangSmith가 데이터셋 전체에 자동 적용합니다. **정답성**과 **근거 기반(환각 방지)** 두 기준을 만듭니다.

In [17]:
def _judge(system, user):
    r = solar.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": user}])
    return "PASS" in r.choices[0].message.content.upper()

# evaluator 시그니처: (run, example) -> {"key","score"}
def correctness(run, example):
    pred = run.outputs.get("answer", "")
    exp = example.outputs.get("expected", "")
    ok = _judge("답이 기대 정답과 의미상 일치하면 PASS, 아니면 FAIL. 한 단어로만.",
                f"질문:{example.inputs['question']}\n기대:{exp}\n답:{pred}")
    return {"key": "correctness", "score": int(ok)}

def groundedness(run, example):
    pred = run.outputs.get("answer", "")
    ok = _judge("답이 근거에 기반하거나 모르면 모른다고 했으면 PASS, 지어냈으면(환각) FAIL. 한 단어로만.",
                f"답:{pred}")
    return {"key": "groundedness", "score": int(ok)}

print("evaluator 2종 정의 완료: correctness / groundedness")

evaluator 2종 정의 완료: correctness / groundedness


## 4. 실험 실행 — evaluate()
`evaluate()`가 데이터셋 전체에 에이전트를 돌리고 evaluator로 채점해 하나의 **실험**으로 묶습니다. UI에서 통과율과 예제별 trace를 함께 볼 수 있습니다.

In [18]:
results = evaluate(
    run_agent,
    data=DATASET,
    evaluators=[correctness, groundedness],
    experiment_prefix="solar-baseline",
)
print("실험 완료 — LangSmith UI에서 통과율과 각 예제 trajectory를 확인하세요.")
print("실험명:", getattr(results, "experiment_name", "(UI 참고)"))

View the evaluation results for experiment: 'solar-baseline-e93171f6' at:
https://smith.langchain.com/o/80ec3eb5-ca32-4d42-95ff-85a0fcfe1810/datasets/b217fc45-8ed8-4c71-83d5-2213a2f4d9f7/compare?selectedSessions=5dc8b0e7-41e4-4ea1-8241-100970df5658




0it [00:00, ?it/s]

실험 완료 — LangSmith UI에서 통과율과 각 예제 trajectory를 확인하세요.
실험명: solar-baseline-e93171f6


## 5. 두 번째 실험과 비교 — 개선/회귀 추적
프롬프트를 바꿔 두 번째 실험을 돌리면 LangSmith가 두 실험의 통과율을 **나란히 비교**합니다. 관측 → 평가 → 개선의 한 사이클을 한 화면에서 돌리는 셈입니다.

In [19]:
@traceable(name="news_agent_strict")
def run_agent_strict(inputs: dict) -> dict:
    q = inputs["question"]
    kw = q.replace("동향", "").replace("요약", "").strip()
    hits = search_news(kw)
    resp = solar.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": "헤드라인에 없는 내용은 절대 추측하지 말고 '확인된 정보 없음'이라고 답하라. 한 문장."},
        {"role": "user", "content": f"질문:{q}\n헤드라인:{hits}"}])
    return {"answer": resp.choices[0].message.content.strip()}

evaluate(run_agent_strict, data=DATASET,
         evaluators=[correctness, groundedness],
         experiment_prefix="solar-strict")
print("→ 두 실험(solar-baseline vs solar-strict)을 UI에서 비교하세요.")

View the evaluation results for experiment: 'solar-strict-e95cdbe1' at:
https://smith.langchain.com/o/80ec3eb5-ca32-4d42-95ff-85a0fcfe1810/datasets/b217fc45-8ed8-4c71-83d5-2213a2f4d9f7/compare?selectedSessions=5018fef4-9497-4602-a283-452410fbc920




0it [00:00, ?it/s]

→ 두 실험(solar-baseline vs solar-strict)을 UI에서 비교하세요.


## 직접 해보기
세 번째 evaluator로 **간결성(conciseness)**을 추가해 보세요. 답변 문장이 2개 이하이면 통과로 채점하는 규칙 기반 evaluator면 됩니다.

In [ ]:
# TODO: conciseness(run, example) 를 만들어 evaluators 리스트에 추가하세요.
#       답변 문장이 2개 이하이면 score=1, 아니면 0.

In [ ]:
# ▼ 정답 예시 — 직접 해본 뒤 주석을 풀고 실행하세요
# def conciseness(run, example):
#     pred = run.outputs.get("answer", "")
#     sents = [s for s in pred.replace("?", ".").replace("!", ".").split(".") if s.strip()]
#     return {"key": "conciseness", "score": int(len(sents) <= 2)}
# evaluate(run_agent, data=DATASET,
#          evaluators=[correctness, groundedness, conciseness],
#          experiment_prefix="solar-conciseness")

## 정리
- OTel **수동 계측**과 달리, `wrap_openai`+`@traceable`로 trajectory가 **자동으로** trace 트리에 기록됩니다.
- 관측에서 멈추지 않고 **데이터셋 + evaluator + `evaluate()`**로 평가·실험 비교까지 한 플랫폼에서 처리했습니다.
- 직접 만든 OTel 계측은 '원리 이해', LangSmith 같은 플랫폼은 '운영 생산성' — 둘을 함께 쓰면 가장 강력합니다.

> 참고: LangSmith SDK는 버전에 따라 `evaluate`/`create_examples` 시그니처가 다를 수 있습니다. 오류 시 최신 문서를 확인하세요.